In [ ]:
from eykthyr.eykthyr import Eykthyr, load_anndata
import scanpy as sc
from eykthyr import pl

# Training Eykthyr and visualizing results

This notebook covers the second and third parts of the Eykthyr pipeline:

1. **Training** — computing TF activity from ArchR outputs, inferring TF → metagene
   regulatory edge weights, and simulating TF perturbations.
2. **Visualization** — PAGA graph layout, spatial flow fields, and developmental
   perturbation strength scoring.

**Prerequisites:** Run the RNA preprocessing notebook first to produce
`spatialatacrna.h5ad`.

**Inputs required:**
- `spatialatacrna.h5ad` — saved session from the RNA preprocessing notebook.
- `data/spatialATACRNAmouseembryo2_peaks.tsv` — ArchR peak × cell matrix.
- `data/spatialATACRNAmouseembryo2_motifs.tsv` — ArchR peak × motif matrix.

In [ ]:
e = load_anndata('spatialatacrna.h5ad')

## Step 1 — Reload preprocessed session

Load the Eykthyr session saved by the RNA preprocessing notebook.
This restores the RNA AnnData, Popari model, and all metadata.

In [ ]:
e.compute_TF_activity(
    peak_tsvs=['data/spatialATACRNAmouseembryo2_peaks.tsv'],
    archr_dataset_names=['spatialATACRNAmouseembryo2'],
    motif_tsvs=['data/spatialATACRNAmouseembryo2_motifs.tsv'],
)

## Step 2 — Compute TF activity

`compute_TF_activity` combines the ArchR peak × cell matrix with the ArchR
peak × motif binary matrix to produce a cell × TF activity score matrix.

Each TF's score is the dot product of peak accessibility with the binary motif
presence vector, then normalized to [0, 1] across cells.  Cells absent from
the ArchR output (e.g. low-quality barcodes filtered by ArchR) are imputed with
the mean TF activity of the remaining cells.

You need to provide, for each dataset:
- `peak_tsvs` — path to the space-delimited peak × cell TSV.
- `archr_dataset_names` — the ArchR project sample name (used to strip the
  `{name}#` prefix from barcodes).
- `motif_tsvs` — path to the space-delimited peak × motif binary TSV.

In [ ]:
e.compute_TF_metagene_weights()

## Step 3 — Infer TF → metagene edge weights

`compute_TF_metagene_weights` runs a spatial sliding-window ridge regression
for each of the `K` metagenes.  For every cell, a neighborhood of spatially
proximal cells (controlled by `num_hops`) is assembled and ridge regression
is used to predict metagene expression from TF activity.  The resulting
regression coefficients are the cell-level TF → metagene edge weights.

This is the most computationally intensive step (~1–2 min per metagene on CPU
with ~2000 cells).

In [ ]:
e.run_all_perturbations()

## Step 4 — Simulate TF perturbations

`run_all_perturbations` simulates TF knockout for every TF in the dataset.
For each TF it:

1. Zeros out that TF's activity column.
2. Propagates the change through the TF → metagene → gene decoder to predict
   post-perturbation gene expression.
3. Stores the result as an AnnData in `perturbed_X`.

The `original_leiden` cluster labels are copied to `perturbed_X[i].obs`
so that perturbation plots can be coloured by the original Leiden clusters.

In [ ]:
e.save_anndata('spatialatacrna.h5ad')

## Visualization

The next cells build a PAGA graph layout and visualize how TF knockout shifts
cell identities across the tissue.

### Step 6 — Prepare PAGA graph layout

`prep_paga` computes UMAP and force-directed graph embeddings on
`perturbed_X`, then fits a PAGA graph to visualize cluster connectivity.
The force-directed layout (`X_draw_graph_fr`) is used alongside the spatial
coordinates in the simulation plots.

In [ ]:
e = load_anndata('spatialatacrna.h5ad')

In [ ]:
pl.prep_paga(e, 'original_leiden')

In [ ]:
pl.paga_spatial_simulation(e, ['Msx1'], 'original_leiden')

### Step 7 — Plot spatial simulation flow

`paga_spatial_simulation` visualizes the TF knockout simulation as a flow
field overlaid on the spatial tissue map and the force-directed graph.
Arrows indicate the predicted direction and magnitude of cell-identity shift
under the TF knockout.  The right panel shows the null-model (randomized) flow
for comparison.

In [ ]:
sc.pp.neighbors(e.perturbed_X[0], use_rep='spatial', key_added='spatial_neighbors')
nns = sc.Neighbors(e.perturbed_X[0], neighbors_key='spatial_neighbors')
nns.compute_neighbors(knn=False, use_rep='spatial', method='gauss')
ventricle_cells = e.perturbed_X[0][e.perturbed_X[0].obs['original_leiden'] == '7']
e.perturbed_X[0].obs['ventricle_distance'] = nns.distances[:,e.perturbed_X[0].obs['original_leiden'] == '7'].min(axis=1)

### Step 8 — Compute pseudotime proxy

We use Gaussian-kernel spatial distance to the ventricular zone (Leiden cluster 7)
as a pseudotime proxy.  This gives each cell a `ventricle_distance` value that
increases with distance from the ventricle — a proxy for differentiation stage.

In [ ]:
ips = pl.development_simulation(e, ['Msx1'])

### Step 9 — Developmental perturbation strength

`development_simulation` computes the inner product of each cell's simulated
flow vector with the pseudotime gradient direction.  A positive inner product
(perturbation strength > 0) means the TF knockout pushes cells *toward* more
differentiated states; negative means it pushes them *toward* less differentiated
(more progenitor-like) states.

The function returns a list of `(TF_name, total_perturbation_strength)` tuples
that can be used to rank TFs by their developmental influence.